In [ ]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

# Funções básicas

In [ ]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale

def load_metadata(video_path):
    path = video_path.replace(".mp4", "_meta.json")

    full_path = os.path.join("/home/guilherme_monteiro/projetos/tcc/data/metadata", os.path.basename(path))

    with open(full_path, "r") as f:
        return json.load(f)

def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    # aplica padding
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    # máscara face interna
    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    # máscara expandida
    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    # borda = expandida - interna
    border_mask = expanded_mask - face_mask

    # fundo = resto
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Funções pra visualização

In [ ]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# Ruido Residual

In [ ]:
def compute_residual(gray):

    # 9: tamanho do kernel 
    # sigmaColor: quanto mistura intensidade (ruído)
    # sigmaSpace: quanto mistura espaço

    smooth = cv2.bilateralFilter(gray, 9, 75, 75)
    residual = gray.astype(np.float32) - smooth.astype(np.float32)
    return residual

In [ ]:
def compute_noise_region(residual, mask):

    values = residual[mask == 1]

    if len(values) < 25:
        return None

    # variância
    variance = np.var(values)

    # energia do ruído 
    energy = np.mean(np.abs(values))

    # entropia 
    hist, _ = np.histogram(values, bins=32)
    hist = hist.astype(np.float32)
    hist = hist / (np.sum(hist) + 1e-6)
    entropy = -np.sum(hist * np.log(hist + 1e-6))

    # kurtosis (CORRIGIDA)
    mean = np.mean(values)
    if variance < 1e-6:
        kurtosis = 0
    else:
        kurtosis = np.mean((values - mean)**4) / (variance**2)

    return {
        "variance": variance,
        "energy": energy,
        "entropy": entropy,
        "kurtosis": kurtosis
    }

def noise_region_differences(f1, f2, prefix):

    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_noise_var_diff": abs(f1["variance"] - f2["variance"]),
        f"{prefix}_noise_energy_diff": abs(f1["energy"] - f2["energy"]),
        f"{prefix}_noise_entropy_diff": abs(f1["entropy"] - f2["entropy"]),
        f"{prefix}_noise_kurtosis_diff": abs(f1["kurtosis"] - f2["kurtosis"]),
    }

def compute_noise_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    residual = compute_residual(gray)

    regions = create_face_regions(frame_std, bbox)

    face = compute_noise_region(residual, regions["face"])
    border = compute_noise_region(residual, regions["border"])
    bg = compute_noise_region(residual, regions["background"])

    features = {}

    # diretas
    if face:
        features["noise_face_variance"] = face["variance"]
        features["noise_face_entropy"] = face["entropy"]
        features["noise_face_energy"] = face["energy"]

    # derivadas
    features.update(noise_region_differences(face, bg, "face_bg"))
    features.update(noise_region_differences(face, border, "face_border"))
    features.update(noise_region_differences(border, bg, "border_bg"))

    return features, residual

## Debbug por video

In [ ]:
def residual_visual(residual):

    res = residual.copy()

    # aumenta contraste do ruído
    res = np.clip(res * 3, -255, 255)

    res_abs = np.abs(res)

    res_vis = cv2.normalize(res_abs, None, 0, 255, cv2.NORM_MINMAX)

    return res_vis.astype(np.uint8)

def debug_noise_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando Residual Noise...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        features, residual = compute_noise_metrics(frame, bbox_scaled)

        res_vis = residual_visual(residual)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",
            f"Face Var: {features.get('noise_face_variance', 0):.4f}",
            f"Face Entropy: {features.get('noise_face_entropy', 0):.3f}",
            f"Face-BG VarianceDiff: {features.get('face_bg_noise_var_diff', 0):.4f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(res_vis, cv2.COLOR_GRAY2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [ ]:
#debug_noise_video(df_true.iloc[1]["video_path"], max_frames=200)

In [ ]:
#debug_noise_video(df_fake.iloc[2]["video_path"], max_frames=200)

In [ ]:
#debug_noise_video(df_fake.iloc[1]["video_path"], max_frames=200)

# Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):
        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        features_noise, _ = compute_noise_metrics(frame_std, bbox_scaled)

        features = {**features_noise}
        features["frame"] = i

        all_features.append(features)

    df = pd.DataFrame(all_features)
    df.insert(0, "video_name", video_name)
    if label is not None:
        df.insert(1, "label", label)

    return df